In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/Lenovo/Downloads/PL_team_matches.csv")

print(df.shape)
df.head()

(3800, 183)


,MatchID,Season,Date,Time,Team,Opponent,HomeAway,Goals,GoalsConceded,HTGoals,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,1,2021/2022,2021-08-13,20:00,Brentford,Arsenal,Home,2,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2021/2022,2021-08-13,20:00,Arsenal,Brentford,Away,0,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2021/2022,2021-08-14,12:30,Man United,Leeds,Home,5,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,2021/2022,2021-08-14,12:30,Leeds,Man United,Away,1,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,2021/2022,2021-08-14,15:00,Burnley,Brighton,Home,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df.sample(5, random_state=42)
df.info()
df.describe(include="all").T
df.isna().mean().sort_values(ascending=False)
df.nunique().sort_values()
df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3800 entries, 0 to 3799
Columns: 183 entries, MatchID to LBCA
dtypes: float64(138), int64(33), object(12)
memory usage: 5.3+ MB


0

In [3]:
print(df.isna().mean().sort_values(ascending=False).head(15))

CLCA    0.863158
CLCD    0.863158
CLCH    0.863158
LBCA    0.852105
LBCD    0.852105
LBCH    0.852105
CLH     0.851579
CLD     0.851579
CLA     0.851579
LBA     0.849474
LBH     0.849474
LBD     0.849474
1XBH    0.804737
1XBA    0.804737
1XBD    0.804737
dtype: float64


In [4]:
print(df.nunique().sort_values().head(10))

Div            1
HomeAway       2
HTR            3
Reds           3
FTR            3
AR             3
HR             3
RedsAgainst    3
Season         5
HTAG           6
dtype: int64


In [5]:
cols_to_keep = [
    'MatchID', 'Date', 'HomeTeam', 'AwayTeam', 'HomeAway', 'FTR', # FTR is the full time result
    'FTHG', 'FTAG', # Home Goals and Away Goals
    'HS', 'AS',     # Shots
    'HST', 'AST'    # Shots on Target
]

In [6]:
df_audit = df[cols_to_keep].copy()
df_audit['Target'] = (df_audit['FTHG'] >= 2).astype(int)

In [7]:
print(df_audit['Target'].value_counts(normalize=True))

Target
0    0.528421
1    0.471579
Name: proportion, dtype: float64


In [8]:
print(df_audit.isna().sum()) # Missing values

MatchID     0
Date        0
HomeTeam    0
AwayTeam    0
HomeAway    0
FTR         0
FTHG        0
FTAG        0
HS          0
AS          0
HST         0
AST         0
Target      0
dtype: int64


In [9]:
print("Unique matches:", df["MatchID"].nunique(), "| Total rows:", len(df))

Unique matches: 1900 | Total rows: 3800


In [10]:
df_model = df[df["HomeAway"] == "Home"].copy()
df_audit['Target'] = (df_audit['FTHG'] >= 2).astype(int)
print("I am filtering the dataset to Home team rows only. I am prediecting wheter the home team will score 2+ goals and keeping away rows would cause duplicates")

I am filtering the dataset to Home team rows only. I am prediecting wheter the home team will score 2+ goals and keeping away rows would cause duplicates


In [11]:
print(df_audit.duplicated().sum())  

0


In [12]:
# The shots on target should never be bigger than the total shots by a team
print("HST > HS (should be 0):", (df_audit['HST'] > df_audit['HS']).sum())
print("AST > AS (should be 0):", (df_audit['AST'] > df_audit['AS']).sum())

HST > HS (should be 0): 0
AST > AS (should be 0): 2


In [13]:
# Now I can find this exact match where the impossible value occurs
bad_rows = df_audit[df_audit['AST'] > df_audit['AS']]
bad_rows[['MatchID', 'Date', 'HomeTeam', 'AwayTeam', 'AS', 'AST']]
print("This is another reason why it makes more sense to just predict whether FTHG >= 2")

This is another reason why it makes more sense to just predict whether FTHG >= 2
